Dbutils in Pyspark

In [0]:
dbutils.fs.mkdirs("/Volumes/pysparkcatalog/pyspark_schema/employee_data/newFolder")

In [0]:
display(dbutils.fs.ls("/Volumes/pysparkcatalog/pyspark_schema/employee_data"))

Widgets


In [0]:
dbutils.widgets.text("department", "SS")
dbutils.widgets.text("Book", "KKK")

select_department=dbutils.widgets.get("department")
sencond=dbutils.widgets.get("Book")



In [0]:
print(select_department)
print(sencond)

# Write into diff file formats in pyspark....
Creating files with formats

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,FloatType

basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/csv"

dbutils.fs.mkdirs(basepath)

#sample data

employee_data=[
    (1,"Amit", "IT",60000,28, "Pune"),
    (2,"Priya", "HR",50000,25, "Mumbai"),
    (3,"Ravi", "IT",70000,30, "Delhi"),
    (4,"Priyanka", "HR",55000,27, "Bangalore"),
    (5,"Rajesh", "IT",65000,29, "Chennai"),
    (6,"Priyanka", "HR",55000,27, "Bangalore"),
        
    ]

employee_columns=["id", "name","department","salary","age","city"]

df=spark.createDataFrame(employee_data, employee_columns)
df.display()


# Wrte the data in csv

df.write.mode("overwrite").option("header", True).csv(basepath)


Write in json

In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/json"

dbutils.fs.mkdirs(basepath)

df.write.mode("overwrite").json(basepath)

Create parquet format file

In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/parquet"

dbutils.fs.mkdirs(basepath)

df.write.mode("overwrite").parquet(basepath)

Orc format  
Column oriented format

In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/ocr"

dbutils.fs.mkdirs(basepath)

df.write.mode("overwrite").orc(basepath)


Wrtie in txt file


In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/txt"

dbutils.fs.mkdirs(basepath)

df.selectExpr("concat_ws(',', id, name, department, salary, age, city) as line")\
    .write.mode("overwrite").text(basepath)

csv, json, txt- does not store schema internally
parquet, orc, delta- they stores schema internally


## Reading files DBFS


In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/csv"


df_csv=spark.read.option("header", True).option('inferSchema', True).csv(basepath)

display(df_csv)

In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/json"


df_json=spark.read.json(basepath)

display(df_json)

In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/parquet"

df_parquet=spark.read.parquet(basepath)
display(df_parquet)
df_parquet.printSchema()

In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/ocr"


df_orc=spark.read.orc(basepath)
display(df_orc)


In [0]:
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/txt"


df_txt=spark.read.text(basepath)
df.display()

Reading messy data csv file


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,FloatType
basepath="/Volumes/pysparkcatalog/pyspark_schema/creating_newfiles/messydata/"
messy_schema=StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", FloatType(), True),
    StructField("_corrupt_record", StringType(),True)
])

df_corrupt=(
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(messy_schema)
    .csv(basepath)
)
df_corrupt.show()
